#### Project 3: Autonomous Media Inclusivity Research Agent
##### Day 3: ReAct Agent — Tools and Autonomous Research

**Goal:** Build and test the autonomous research agent that orchestrates all 4 tools to produce evidence-based inclusivity findings.

---

### Architectural Note: Why ReAct Instead of a 5-Node Graph

Day 1 planned a fixed 5-node LangGraph workflow. During implementation,
the ReAct (Reason + Act) pattern proved a better fit for this task:

| Approach | Fixed 5-Node Graph | ReAct Agent (chosen) |
|---|---|---|
| Tool call order | Hardcoded sequence | Agent decides dynamically |
| Retry on failure | Manual logic | Built into the loop |
| Depth of research | Fixed per node | Agent calls tools as many times as needed |
| LangGraph version | Requires custom StateGraph | `create_react_agent` — one line |

The ReAct loop: **Thought → Action (tool call) → Observation → Thought → ...**  
The agent keeps reasoning until it has enough evidence to write the full research summary.

---

### What Day 3 Builds

```
Tool 1: rag_query_tool     → Pinecone knowledge base (industry benchmarks)
Tool 2: search_wikipedia   → Company background, ownership, editorial history  
Tool 3: search_newsapi     → What critics say about the outlet's diversity record
Tool 4: analyse_rss_feed   → Primary source: today's bylines, language, coverage
        |
        v
create_react_agent (LangGraph prebuilt)
        |
        v
Final research summary → passed to Day 4 (report generation)
```

In [ ]:
# CELL 1: Setup — imports and working directory
# Must run from project root so src/ imports resolve correctly

import os
import sys
from dotenv import load_dotenv

# Ensure working directory is project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Add project root to path so src/ imports work
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

load_dotenv()

print(f"Working directory: {os.getcwd()}")
print(f"Python path includes project root: {os.getcwd() in sys.path}")
print()
print("Environment check:")
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "NEWSAPI_KEY", "PINECONE_API_KEY"]:
    val = os.getenv(key)
    status = f"LOADED ({val[:8]}...)" if val else "MISSING"
    print(f"  {key}: {status}")

## Part 1: The Four Research Tools

Each tool is a LangChain `@tool` — a Python function decorated so the ReAct agent
can discover, call, and read results from it autonomously.

We test each tool individually before wiring them into the agent.

In [ ]:
# CELL 2: Tool 1 — RAG Knowledge Base Search
# Queries our Pinecone index (built in Day 2) for industry benchmarks.
# The agent uses this to ground its scores in verified research.

from src.tools import rag_query_tool

print(f"Tool name:        {rag_query_tool.name}")
print(f"Tool description: {rag_query_tool.description[:120]}...")
print()

# Test: query with an angle filter
result = rag_query_tool.invoke({
    "query": "how are women represented in newsroom bylines and story selection",
    "angle": "bylines_and_story_selection"
})

print(result)

In [ ]:
# CELL 3: Tool 2 — Wikipedia Search
# Fetches company background: ownership, editorial history, reach.
# Free REST API — no key required.

from src.tools import search_wikipedia

print(f"Tool name:        {search_wikipedia.name}")
print(f"Tool description: {search_wikipedia.description[:120]}...")
print()

result = search_wikipedia.invoke({"query": "Al Jazeera English media company"})
print(result)

In [ ]:
# CELL 4: Tool 3 — NewsAPI Search
# Searches what journalists and critics say about an outlet's diversity record.
# Last 30 days, top 5 most relevant articles.

from src.tools import search_newsapi

print(f"Tool name:        {search_newsapi.name}")
print(f"Tool description: {search_newsapi.description[:120]}...")
print()

result = search_newsapi.invoke({"query": "Al Jazeera English newsroom diversity inclusion representation"})
print(result)

In [ ]:
# CELL 5: Tool 4 — RSS Feed Analyser
# Primary source evidence: fetches today's articles directly from the outlet.
# Returns bylines, community coverage counts, and language flags.
# No API key — reads public RSS feeds.
#
# Al Jazeera: RSS only, ~105 char teasers, 0 named bylines (transparency gap)
# Compare with NPR which has named bylines + full content:encoded body

from src.tools import analyse_rss_feed

print(f"Tool name:        {analyse_rss_feed.name}")
print(f"Tool description: {analyse_rss_feed.description[:120]}...")
print()

result = analyse_rss_feed.invoke({"company": "Al Jazeera English"})
print(result)

## Part 2: Wiring Tools into the ReAct Agent

The ReAct (Reason + Act) pattern lets the LLM decide dynamically:
- **Which tool to call** based on what it still needs to know
- **How many times** to call tools (it keeps going until it has enough evidence)
- **What to conclude** after seeing all the observations

LangGraph's `create_react_agent` implements this loop:

```
Thought: "I need primary source data — I'll call analyse_rss_feed"
Action:  analyse_rss_feed("NPR")
Obs:     PRIMARY SOURCE ANALYSIS: NPR RSS Feed... [30 articles, 27 named bylines, coverage counts]
Thought: "Race had 5 articles but disability had zero — I need benchmarks on disability erasure"
Action:  rag_query_tool("disability representation erasure journalism standards")
Obs:     KNOWLEDGE BASE: ... [benchmark text from Pinecone]
Thought: "Now I need to check what critics say about NPR's specific diversity gaps"
Action:  search_newsapi("NPR disability LGBTQ+ coverage gaps public radio")
Obs:     NEWS SEARCH: ... [journalist perspectives]
...continues until the agent has covered all 4 angles × 4 communities...
Final:   Full research summary
```

In [ ]:
# CELL 6: Build the ReAct agent
# create_react_agent takes the LLM + list of tools and builds the full loop.

from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from src.tools import analyse_rss_feed, rag_query_tool, search_newsapi, search_wikipedia

# LLM for reasoning — gpt-4o-mini balances quality and cost for the tool-calling loop
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# All 4 tools available to the agent
TOOLS = [analyse_rss_feed, rag_query_tool, search_newsapi, search_wikipedia]

# Build the agent — this creates the full ReAct graph internally
agent = create_react_agent(llm, TOOLS)

print("ReAct agent created.")
print(f"LLM model: {llm.model_name}")
print(f"Tools bound: {[t.name for t in TOOLS]}")
print()
print("Agent graph nodes:")
for node in agent.get_graph().nodes:
    print(f"  - {node}")

In [ ]:
# CELL 7: Run the full research pipeline
# This invokes the agent with a structured research prompt.
# The agent will call tools, read results, and iterate until
# it has covered all 4 angles × 4 communities.
#
# Expected: ~8-12 tool calls, ~30-90 seconds to complete.

from src.agents.media_agent import run_research

# NPR: named bylines + full article body via content:encoded — good first full test
COMPANY = "NPR"

print(f"Starting research: {COMPANY}")
print("This will take 30-90 seconds while the agent calls all 4 tools...")
print()

result = run_research(COMPANY)

print(f"\nResearch complete.")
print(f"Total messages in history: {len(result['messages'])}")
print(f"Error: {result['error']}")

In [ ]:
# CELL 8: Inspect the ReAct reasoning trace
# The message history shows every Thought → Action → Observation step.
# This is the evidence that the agent is grounding its conclusions in real data.

messages = result["messages"]

print(f"Full message trace ({len(messages)} messages)")
print("=" * 60)

for i, msg in enumerate(messages):
    msg_type = msg.__class__.__name__
    
    if msg_type == "HumanMessage":
        print(f"\n[{i}] HUMAN (research prompt — truncated)")
        print(str(msg.content)[:200] + "...")
    
    elif msg_type == "AIMessage":
        tool_calls = getattr(msg, "tool_calls", [])
        if tool_calls:
            print(f"\n[{i}] AGENT → TOOL CALLS")
            for tc in tool_calls:
                args_preview = str(tc.get("args", {}))[:100]
                print(f"      {tc['name']}({args_preview})")
        elif msg.content:
            print(f"\n[{i}] AGENT → FINAL ANALYSIS (truncated)")
            print(str(msg.content)[:300] + "...")
    
    elif msg_type == "ToolMessage":
        print(f"\n[{i}] TOOL RESULT: {msg.name}")
        print("      " + str(msg.content)[:200] + "...")

# Count how many tool calls were made
tool_call_count = sum(
    len(getattr(m, "tool_calls", []))
    for m in messages
    if getattr(m, "tool_calls", None)
)
print(f"\n{'='*60}")
print(f"Total tool calls made: {tool_call_count}")

In [ ]:
# CELL 9: View the final research summary
# This is what the agent produced after all its tool calls.
# It will be passed to Day 4's report generator for formatting.

print(f"RESEARCH SUMMARY: {COMPANY}")
print("=" * 60)
print(result["final_analysis"])

In [ ]:
# CELL 10: Quality check — verify all 4 tools were called
# For a valid research run, the agent must have called all 4 tools.
# If any are missing, the research is incomplete.

REQUIRED_TOOLS = {"analyse_rss_feed", "rag_query_tool", "search_newsapi", "search_wikipedia"}

tools_used = set()
for msg in messages:
    for tc in getattr(msg, "tool_calls", []):
        tools_used.add(tc["name"])

rag_calls = sum(
    1 for msg in messages
    for tc in getattr(msg, "tool_calls", [])
    if tc["name"] == "rag_query_tool"
)

print("Research Quality Check")
print("=" * 40)

all_tools_used = True
for tool_name in sorted(REQUIRED_TOOLS):
    used = tool_name in tools_used
    status = "PASS" if used else "FAIL"
    if not used:
        all_tools_used = False
    print(f"{status}  {tool_name}")

print()
print(f"RAG calls (protocol requires >= 4): {rag_calls} — {'PASS' if rag_calls >= 4 else 'FAIL'}")
print(f"Final analysis present:             {'PASS' if result['final_analysis'] else 'FAIL'}")
print(f"No errors:                          {'PASS' if not result['error'] else 'FAIL'}")
print()
print("=" * 40)
if all_tools_used and rag_calls >= 4 and result["final_analysis"] and not result["error"]:
    print("ALL CHECKS PASSED — research is complete and grounded")
else:
    print("SOME CHECKS FAILED — review tool calls above")

## Day 3 Complete — Summary

### What was built:
- **4 LangChain `@tool` functions** — each wraps one research source
- **LangGraph ReAct agent** — autonomously decides which tools to call and when
- **Research quality checks** — verifies all 4 tools were called before accepting results

### Why this approach prevents hallucination:
Every claim in the final analysis is grounded in a tool result visible in the message trace:
- Scores are compared against Pinecone benchmarks (rag_query_tool)
- Byline counts come from live RSS data (analyse_rss_feed)
- Critic quotes come from NewsAPI (search_newsapi)
- Company facts come from Wikipedia (search_wikipedia)

If no tool returned evidence for a claim, the agent was instructed to say so explicitly.

### Monitored outlets and their data quality:

| Outlet | Data source | Bylines | Content depth |
|---|---|---|---|
| The Guardian | Guardian Open Platform API | 30/30 named | Full article body text |
| NPR | RSS content:encoded | Named per article | ~850 chars |
| New York Times | RSS | Named bylines | ~130 char teaser |
| Al Jazeera English | RSS | 0/30 (transparency gap) | ~105 char teaser |

### Next up — Day 4:
- Take the `final_analysis` string from this notebook
- Pass it to the report generator (`src/report_generator.py`)
- Format into the full 11-section Markdown report
- Confirm FastAPI + N8N deliver the same payload that was tested with the mock in Day 2